# Implement with WGF

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import optuna
import jax
import jax.numpy as jnp
import optuna
import json
from functools import partial
from jax import jit, vmap, vmap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np

In [2]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from src.gradient_flows.optimize import objective, load_data_slice, calculate_losses
from src.data_io.maps import load_maps_npz
from src.gradient_flows.utils import extract_quality_trajectories, parse_df_row, prepare_play_data
from src.gradient_flows.viz_tools import run_simulation, create_interactive_plot


In [3]:
%load_ext autoreload
%autoreload 2

## Optimize Variables

In [4]:
df = pd.read_parquet("../data/processed/def_features_test/defense_01.18.2016.GSW.at.CLE_21500622.parquet")
df = df.drop(index=[29, 84], errors='ignore')
df = df.reset_index(drop=True)

In [ ]:
STUDY_NAME = "nba-defensive-optimization-v6-final"
STORAGE_NAME = f"sqlite:///{STUDY_NAME}.db"

print("Starting optimization study...")

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_NAME,
    directions=["minimize", "minimize"],
    load_if_exists=True # Set to False for the first run!
)

# Run 100 trials (adjust this number based on how much time you have)
study.optimize(lambda trial: objective(trial, df), n_trials=500)

print("\nOptimization Finished!")
print(f"Total trials recorded: {len(study.trials)}")

# Print the best trade-offs (Pareto Front)
print("\nPareto Front (Best Trials):")
for trial in study.best_trials:
    print(f"  Trial {trial.number}:")
    print(f"  Losses (Pressure, Smoothness): {trial.values}")
    
# Try to visualize the Pareto front
try:
    import optuna.visualization as vis
    fig = vis.plot_pareto_front(study)
    fig.show()
    
except ImportError:
    print("Skipping visualization. (Install plotly/kaleido to view the Pareto front graph).")

Starting optimization study...


[I 2026-02-28 05:36:08,697] Using an existing study with name 'nba-defensive-optimization-v5-balance' instead of creating a new one.
[I 2026-02-28 05:36:13,284] Trial 500 finished with values: [3.6132750511169434, 0.46213266253471375] and parameters: {'field_weight': -39.772653833585515, 'attraction_weight': 16.03540790549985, 'basket_gravity_weight': 13.734613702226392, 'global_ball_weight': 0.016707356244790257, 'sigma_long': 7.708463879551956, 'sigma_wide': 3.443919369518383, 'cushion_dist': 1.896720638324839, 'lane_blocking_weight': 30.51580650151156, 'occupancy_weight': 42.823223601774, 'cohesion_weight': 0.8321912276931938, 'formation_radius': 13.015426310207433, 'k_softmin': 6.197755207815001, 'jko_lambda': 0.41500985944728414, 'sinkhorn_epsilon': 0.05, 'learning_rate': 0.06297313711266736, 'sprint_penalty_weight': 1.4052640326558865, 'acceleration_penalty_weight': 93.45251026190694}.
[I 2026-02-28 05:36:13,830] Trial 501 finished with values: [3.108164072036743, 0.5317355990409


Optimization Finished!
Total trials recorded: 1000

Pareto Front (Best Trials):
  Trial 288:
  Losses (Pressure, Smoothness): [3.7774198055267334, 0.4134094715118408]
  Trial 295:
  Losses (Pressure, Smoothness): [3.910896062850952, 0.379550963640213]
  Trial 306:
  Losses (Pressure, Smoothness): [3.488588333129883, 0.48777106404304504]
  Trial 346:
  Losses (Pressure, Smoothness): [2.9570581912994385, 0.5300864577293396]
  Trial 367:
  Losses (Pressure, Smoothness): [2.9570581912994385, 0.5300864577293396]
  Trial 392:
  Losses (Pressure, Smoothness): [3.910896062850952, 0.379550963640213]
  Trial 405:
  Losses (Pressure, Smoothness): [3.1033482551574707, 0.5162290930747986]
  Trial 427:
  Losses (Pressure, Smoothness): [3.683873414993286, 0.4391002953052521]
  Trial 429:
  Losses (Pressure, Smoothness): [3.488588333129883, 0.48777106404304504]
  Trial 450:
  Losses (Pressure, Smoothness): [3.944554090499878, 0.37446609139442444]
  Trial 473:
  Losses (Pressure, Smoothness): [3.78512